In [2]:
!pip install cirq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 78.0 MB/s eta 0:00:00



## Geleneksel Kuantum Algoritmalarına Giriş (Deutsch-Jozsa)

Artık kuantum kapılarını ve devreleri nasıl kuracağımızı biliyoruz. Şimdi bu temel araçları kullanarak, klasik hesaplamaya göre hızlanma sağlayabilen ilk basit kuantum algoritması olan **Deutsch-Jozsa Algoritması**'nın mantığını inceleyelim.

### Deutsch-Jozsa Algoritması

**Amaç:** Bilinmeyen bir ikili fonksiyon ($f: \{0, 1\}^n \to \{0, 1\}$) hakkında sadece tek bir kuantum sorgusu ile bilgi edinmek.

Fonksiyon $f(x)$ iki tipten biri olabilir:

1.  **Sabit (Constant):** Tüm girdiler için aynı çıktıyı verir (tüm çıktılar **0**'dır veya tüm çıktılar **1**'dir).
2.  **Dengeli (Balanced):** Girdilerin yarısı için **0**, diğer yarısı için **1** çıktısını verir.

**Klasik Yaklaşım:** Fonksiyonun sabit mi yoksa dengeli mi olduğunu anlamak için, en kötü senaryoda $2^{n-1} + 1$ kez sorgulama yapmanız gerekir. (Örn: 2 qubit için $2^{2-1} + 1 = 3$ kez).

**Kuantum Yaklaşım (Deutsch-Jozsa):** Sadece **tek bir** sorgulama ile (kuantum kapıları serisi) fonksiyonun tipini (Sabit mi, Dengeli mi) kesin olarak bulabiliriz.

### Algoritmanın Ana Adımları

Deutsch-Jozsa, esas olarak iki güçlü kuantum prensibini kullanır:

1.  **Süperpozisyon:** Tüm olası girdileri aynı anda denemek için.
2.  **Faz Geri Beslemesi (Phase Kickback):** Fonksiyonun çıktı bilgisini, girdi qubitlerinin **fazına** geri kodlamak için.

#### Deutsch-Jozsa Devresi (Basitleştirilmiş $n=1$ Qubit için)

1.  **Başlangıç:** İki qubit kullanırız: **Girdi Qubit** ($q_0$) ve **Yardımcı Qubit** ($q_1$). Başlangıç: $|0\rangle|0\rangle$.
2.  **Hazırlık:**
      * $q_0$'ı süperpozisyona getiririz: $H(q_0)$.
      * $q_1$'e $X$ ve $H$ uygulayarak onu negatif süperpozisyona getiririz: $X(q_1)$, $H(q_1)$. (Bu, faz geri beslemesi için gereklidir.)
      * Durum: $\frac{|0\rangle+|1\rangle}{\sqrt{2}} \cdot \frac{|0\rangle-|1\rangle}{\sqrt{2}}$
3.  **Kuantum Oracle (Sorgu):** Fonksiyonu temsil eden kapıyı uygularız ($U_f$). Bu, tek kuantum sorgusudur.
4.  **Son İşlem:** $q_0$'a tekrar $H$ uygularız.
5.  **Ölçüm:** $q_0$'ı ölçeriz.

**Sonuç Yorumu:**

  * Eğer $q_0$ sonucu **0** ise: Fonksiyon **Sabit**'tir.
  * Eğer $q_0$ sonucu **1** ise: Fonksiyon **Dengeli**'dir.

Bu inanılmaz bir sonuçtur çünkü tüm olası girdileri denemeden sonuca ulaşıyoruz\!

**Cirq Kod Örneği: Deutsch-Jozsa (Sabit Fonksiyon İçin)**

$f(x)=0$ sabit fonksiyonunu temsil eden bir oracle kullanalım. (Bu, sadece $U_f = I$ (Birim Matris) demektir, yani hiçbir şey yapmaz.)

In [9]:
import cirq
import numpy as np

q0 = cirq.GridQubit(0, 0) # Girdi Qubit
q1 = cirq.GridQubit(0, 1) # Yardımcı Qubit

# 1. Devre Başlangıcı (Hazırlık)
circuit = cirq.Circuit(
    # Yardımcı q1'i |-> durumuna getir (X |0> -> |1>, sonra H)
    cirq.X(q1),
    cirq.H(q0),
    cirq.H(q1),
)

# 2. Kuantum Oracle (Uf): f(x)=0 SABİT fonksiyon
# Sabit 0 fonksiyonu hiçbir işlem yapmaz (I)
# Bu kısım boş bırakılır veya cirq.I(q0), cirq.I(q1) eklenebilir.
# circuit.append([cirq.I(q0), cirq.I(q1)]) # İsteğe bağlı, matris çarpımı için

# 3. Son İşlem ve Ölçüm
circuit.append([
    cirq.H(q0), # Girdi Qubit'e tekrar H uygula
    cirq.measure(q0, key='result')
])


# Simülasyon
simulator = cirq.Simulator()
results = simulator.run(circuit, repetitions=100)

print("### Deutsch-Jozsa Devresi (Sabit f(x)=0) ###")
print(circuit)

print("\n### Ölçüm Sonuçları (100 Çalıştırma) ###")
print("q0 ölçümünün 0 olması, fonksiyonun SABİT olduğunu gösterir.")
print(results.histogram(key='result'))

### Deutsch-Jozsa Devresi (Sabit f(x)=0) ###
(0, 0): ───H───H───M('result')───

(0, 1): ───X───H─────────────────

### Ölçüm Sonuçları (100 Çalıştırma) ###
q0 ölçümünün 0 olması, fonksiyonun SABİT olduğunu gösterir.
Counter({0: 100})


#### Çıktı Yorumu

Çıktıda, $q_0$ ölçüm sonucunun neredeyse tamamen **0** olduğunu göreceksiniz. Algoritma bize doğru bir şekilde, **tek bir sorgu** ile, fonksiyonun **Sabit** olduğunu söyledi\!
